# Find a user collection and Explore it

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-10

- `fichier bps_dp2cosmos.sh`

```bash
export VISITIDS=$(paste -sd, cosmos_small.txt)
export CCDIDS=$(paste -sd, ccdIds.txt)

release="w_2026_40"

echo "run BPS"
bps_submit.sh --interactive bps_dp2cosmos_parsl.yaml
```

- `fichier bps_dp2cosmos_parsl.yaml`
```
pipelineYaml: "${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit"

payload:
  runInit: true
  payloadName: dp2cosmos202610
  butlerConfig: main
  inCollection: LSSTCam/defaults
  output: u/${USER}/LSSTCam/runcosmos01_20261008_RunCosmosDemo/stage1
  dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND detector IN (${CCDIDS}) AND exposure IN (${VISITIDS}) AND visit IN (${VISITIDS})"
```

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graph : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/

## Import

In [1]:
import warnings

warnings.filterwarnings(
    "error",
    category=FutureWarning,
    message=".*Butler.collections.*",
)

In [2]:
from lsst.daf.butler import Butler

butler_url = "main"
butler = Butler(butler_url)

## Identify existing user collections

In [3]:
from lsst.daf.butler import CollectionType

prefix = "u/dagoret/"

collections = list(butler.collections.query(f"{prefix}*"))

for collection in collections:
    print(collection)

print(f"\nNombre de collections : {len(collections)}")


u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage1/20261008T151954Z
u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage1

Nombre de collections : 2


## 

In [12]:
collection_path = (
    "u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/"
    "stage1/20261008T151954Z"
)

exists = any(
    name == collection_path
    for name in butler.collections.query(collection_path)
)
print(f"exists = {exists} for {collection_path}")

if collection_path in butler.collections.query(collection_path):
    info = butler.collections.get_info(collection_path)
    print(f"Collection trouvée : {collection_path}")
    print(f"Type : {info.type}")
else:
    print(f"Collection inexistante : {collection_path}")

exists = True for u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage1/20261008T151954Z
Collection trouvée : u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage1/20261008T151954Z
Type : 1


## Find datasets in my collection

In [13]:
import collections

### Total number of datasets

In [18]:
refs = butler.registry.queryDatasets(
    datasetType="*",
    collections=collection_path,
    findFirst=False,
)

refs = list(refs)

print(f"Total number of datasets referenced  : {len(refs)} in {collection_path}")

counts = collections.Counter(
    ref.datasetType.name for ref in refs
)

for name, count in sorted(counts.items()):
    print(f"{name:40s} {count:8d}")

Total number of datasets referenced  : 3456 in u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage1/20261008T151954Z
analyzeSingleVisitStarAssociation_config        1
analyzeSingleVisitStarAssociation_log          10
analyzeSingleVisitStarAssociation_metadata       10
associateIsolatedStar_config                    1
associateIsolatedStar_log                      10
associateIsolatedStar_metadata                 10
calibrateImage_config                           1
calibrateImage_log                            179
calibrateImage_metadata                       179
consolidateSingleVisitStar_config               1
consolidateSingleVisitStar_log                  1
consolidateSingleVisitStar_metadata             1
consolidateVisitSummary_config                  1
consolidateVisitSummary_log                     1
consolidateVisitSummary_metadata                1
initial_astrometry_match_detector             179
initial_photometry_match_detector             179
isolated_star                

### Examine a number of references

In [16]:
for ref in refs[:20]:
    print(
        ref.datasetType.name,
        ref.dataId
    )

isr_config {}
isrStatistics {instrument: 'LSSTCam', detector: 112, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
isrStatistics {instrument: 'LSSTCam', detector: 128, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
isrStatistics {instrument: 'LSSTCam', detector: 38, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
isrStatistics {instrument: 'LSSTCam', detector: 91, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
isrStatistics {instrument: 'LSSTCam', detector: 144, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
isrStatistics {instrument: 'LSSTCam', detector: 4, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filt

### Examine post_isr_image

In [17]:
for ref in butler.registry.queryDatasets(
    datasetType="post_isr_image",
    collections=collection_path,
    findFirst=False,
):
    print(ref.dataId)

{instrument: 'LSSTCam', detector: 87, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 145, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 55, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 41, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 59, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 108, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 16, exposure: 2025123100242, band: 'i', day_obs: 202

Dans ta collection, il y a plusieurs catalogues liés aux étoiles. Pour le pipeline Rubin que tu as exécuté, les trois types les plus intéressants sont les suivants.


- Dataset type	    :->  Contenu / rôle 

- single_visit_star	: ->  Catalogue consolidé des étoiles détectées sur plusieurs visites, produit final de l'association des étoiles sur une visite unique.

- single_visit_star_detector :->	Catalogue des étoiles associé aux détecteurs individuels (179 datasets dans ta collection).

- isolated_star	    :-> Échantillons d'étoiles isolées, utilisés notamment pour la calibration photométrique et astrométrique.


- isolated_star_association	:-> Résultats de l'association des étoiles isolées entre observations.

  
- single_visit_psf_star	    :-> Étoiles sélectionnées pour la modélisation de la PSF, une sélection spécifique et non un catalogue stellaire général.


Je te conseille de commencer par single_visit_star, puis de regarder single_visit_star_detector si tu souhaites étudier les étoiles détectées sur chaque CCD.


### 1. Lire le catalogue single_visit_star

In [19]:
refs = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs))

for ref in refs:
    print(ref.dataId)

Nombre de catalogues : 1
{instrument: 'LSSTCam', visit: 2025123100242, band: 'i', day_obs: 20251231, physical_filter: 'i_39'}


### 2. Charger le catalogue dans un DataFrame

In [23]:
import pandas as pd

ref = refs[0]

catalog = butler.get(ref)

print(type(catalog))
print(len(catalog))
#print(catalog.schema)

<class 'astropy.table.table.Table'>
197865


In [25]:
catalog

coord_ra,coord_dec,parentSourceId,x,y,xErr,yErr,ra,dec,raErr,decErr,ra_dec_Cov,calibFlux,calibFluxErr,ap03Flux,ap03FluxErr,ap03Flux_flag,ap06Flux,ap06FluxErr,ap06Flux_flag,ap09Flux,ap09FluxErr,ap09Flux_flag,ap12Flux,ap12FluxErr,ap12Flux_flag,ap17Flux,ap17FluxErr,ap17Flux_flag,ap25Flux,ap25FluxErr,ap25Flux_flag,ap35Flux,ap35FluxErr,ap35Flux_flag,ap50Flux,ap50FluxErr,ap50Flux_flag,ap70Flux,ap70FluxErr,ap70Flux_flag,sky,skyErr,psfFlux,psfFluxErr,ixx,iyy,ixy,ixxPSF,iyyPSF,ixyPSF,ixxDebiasedPSF,iyyDebiasedPSF,ixyDebiasedPSF,gaussianFlux,gaussianFluxErr,extendedness,sizeExtendedness,blendedness_abs,blendedness_flag,blendedness_flag_noCentroid,blendedness_flag_noShape,apFlux_12_0_flag,apFlux_12_0_flag_apertureTruncated,apFlux_12_0_flag_sincCoeffsTruncated,apFlux_12_0_instFlux,apFlux_12_0_instFluxErr,apFlux_17_0_flag,apFlux_17_0_instFlux,apFlux_17_0_instFluxErr,apFlux_35_0_flag,apFlux_35_0_instFlux,apFlux_35_0_instFluxErr,apFlux_50_0_flag,apFlux_50_0_instFlux,apFlux_50_0_instFluxErr,normCompTophatFlux_flag,normCompTophatFlux_instFlux,normCompTophatFlux_instFluxErr,extendedness_flag,sizeExtendedness_flag,footprintArea_value,invalidPsfFlag,jacobian_flag,jacobian_value,localBackground_instFlux,localBackground_instFluxErr,localBackground_flag,localBackground_flag_noGoodPixels,localBackground_flag_noPsf,pixelFlags_bad,pixelFlags_cr,pixelFlags_crCenter,pixelFlags_edge,pixelFlags_interpolated,pixelFlags_interpolatedCenter,pixelFlags_nodata,pixelFlags_offimage,pixelFlags_saturated,pixelFlags_saturatedCenter,pixelFlags_suspect,pixelFlags_suspectCenter,psfFlux_apCorr,psfFlux_apCorrErr,psfFlux_area,psfFlux_flag,psfFlux_flag_apCorr,psfFlux_flag_edge,psfFlux_flag_noGoodPixels,gaussianFlux_flag,centroid_flag,centroid_flag_almostNoSecondDerivative,centroid_flag_badError,centroid_flag_edge,centroid_flag_noSecondDerivative,centroid_flag_notAtMaximum,centroid_flag_resetToPeak,variance_flag,variance_flag_emptyFootprint,variance_value,calib_astrometry_used,calib_photometry_reserved,calib_photometry_used,calib_psf_candidate,calib_psf_reserved,calib_psf_used,deblend_deblendedAsPsf,deblend_hasStrayFlux,deblend_masked,deblend_nChild,deblend_parentTooBig,deblend_patchedTemplate,deblend_rampedTemplate,deblend_skipped,deblend_tooManyPeaks,hsmPsfMoments_flag,hsmPsfMoments_flag_no_pixels,hsmPsfMoments_flag_not_contained,hsmPsfMoments_flag_parent_source,iDebiasedPSF_flag,iDebiasedPSF_flag_no_pixels,iDebiasedPSF_flag_not_contained,iDebiasedPSF_flag_parent_source,iDebiasedPSF_flag_galsim,iDebiasedPSF_flag_edge,hsmShapeRegauss_flag,hsmShapeRegauss_flag_galsim,hsmShapeRegauss_flag_no_pixels,hsmShapeRegauss_flag_not_contained,hsmShapeRegauss_flag_parent_source,sky_source,detect_isPrimary,visit,detector,band,physical_filter,sourceId
float64,float64,int64,float64,float64,float32,float32,float64,float64,float32,float32,float32,float64,float64,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,bool,bool,bool,bool,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,float64,float64,bool,bool,int32,bool,bool,float64,float64,float64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,float64,float64,float32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,float64,bool,bool,bool,bool,bool,bool,bool,bool,bool,int32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,int64,int16,str1,str4,int64
149.72354490306768,4.094282559929577,0,1684.9626314194143,109.0030071428328,0.007609739,0.008757597,149.72354490306768,4.094282559929577,4.2397963e-07,4.837507e-07,9.00185e-15,399776.74895586516,1439.215745396813,209741.90020350707,542.411195398948,False,341183.3780161533

In [26]:
df = catalog.to_pandas()

display(df.head())
print(df.columns.tolist())

,coord_ra,coord_dec,parentSourceId,x,y,xErr,yErr,ra,dec,raErr,...,hsmShapeRegauss_flag_no_pixels,hsmShapeRegauss_flag_not_contained,hsmShapeRegauss_flag_parent_source,sky_source,detect_isPrimary,visit,detector,band,physical_filter,sourceId
0,149.723545,4.094283,0,1684.962631,109.003007,0.007610,0.008758,149.723545,4.094283,4.239796e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595137
1,149.732280,4.090574,0,1850.407945,151.167299,0.005548,0.006468,149.732280,4.090574,3.092431e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595139
2,149.724178,4.089119,0,1710.549141,199.373134,0.019675,0.022984,149.724178,4.089119,1.096717e-06,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595140
3,149.822947,4.060673,0,3543.564555,435.476449,0.006972,0.006810,149.822947,4.060673,3.868239e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595141
4,149.733416,4.070376,0,1926.690468,508.331023,0.006517,0.006940,149.733416,4.070376,3.623014e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595143


['coord_ra', 'coord_dec', 'parentSourceId', 'x', 'y', 'xErr', 'yErr', 'ra', 'dec', 'raErr', 'decErr', 'ra_dec_Cov', 'calibFlux', 'calibFluxErr', 'ap03Flux', 'ap03FluxErr', 'ap03Flux_flag', 'ap06Flux', 'ap06FluxErr', 'ap06Flux_flag', 'ap09Flux', 'ap09FluxErr', 'ap09Flux_flag', 'ap12Flux', 'ap12FluxErr', 'ap12Flux_flag', 'ap17Flux', 'ap17FluxErr', 'ap17Flux_flag', 'ap25Flux', 'ap25FluxErr', 'ap25Flux_flag', 'ap35Flux', 'ap35FluxErr', 'ap35Flux_flag', 'ap50Flux', 'ap50FluxErr', 'ap50Flux_flag', 'ap70Flux', 'ap70FluxErr', 'ap70Flux_flag', 'sky', 'skyErr', 'psfFlux', 'psfFluxErr', 'ixx', 'iyy', 'ixy', 'ixxPSF', 'iyyPSF', 'ixyPSF', 'ixxDebiasedPSF', 'iyyDebiasedPSF', 'ixyDebiasedPSF', 'gaussianFlux', 'gaussianFluxErr', 'extendedness', 'sizeExtendedness', 'blendedness_abs', 'blendedness_flag', 'blendedness_flag_noCentroid', 'blendedness_flag_noShape', 'apFlux_12_0_flag', 'apFlux_12_0_flag_apertureTruncated', 'apFlux_12_0_flag_sincCoeffsTruncated', 'apFlux_12_0_instFlux', 'apFlux_12_0_instFlux

### 3. Examiner les étoiles détectées par détecteur

Pour accéder aux 179 catalogues single_visit_star_detector :

In [28]:
refs_detector = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs_detector))

catalog_detector = butler.get(refs_detector[0])

print(type(catalog_detector))


Nombre de catalogues : 179
<class 'astropy.table.table.Table'>


In [29]:
df_det = catalog.to_pandas()

display(df_det.head())
print(df_det.columns.tolist())

,coord_ra,coord_dec,parentSourceId,x,y,xErr,yErr,ra,dec,raErr,...,hsmShapeRegauss_flag_no_pixels,hsmShapeRegauss_flag_not_contained,hsmShapeRegauss_flag_parent_source,sky_source,detect_isPrimary,visit,detector,band,physical_filter,sourceId
0,149.723545,4.094283,0,1684.962631,109.003007,0.007610,0.008758,149.723545,4.094283,4.239796e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595137
1,149.732280,4.090574,0,1850.407945,151.167299,0.005548,0.006468,149.732280,4.090574,3.092431e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595139
2,149.724178,4.089119,0,1710.549141,199.373134,0.019675,0.022984,149.724178,4.089119,1.096717e-06,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595140
3,149.822947,4.060673,0,3543.564555,435.476449,0.006972,0.006810,149.822947,4.060673,3.868239e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595141
4,149.733416,4.070376,0,1926.690468,508.331023,0.006517,0.006940,149.733416,4.070376,3.623014e-07,...,False,False,False,False,True,2025123100242,1,i,i_39,25697818245595143


['coord_ra', 'coord_dec', 'parentSourceId', 'x', 'y', 'xErr', 'yErr', 'ra', 'dec', 'raErr', 'decErr', 'ra_dec_Cov', 'calibFlux', 'calibFluxErr', 'ap03Flux', 'ap03FluxErr', 'ap03Flux_flag', 'ap06Flux', 'ap06FluxErr', 'ap06Flux_flag', 'ap09Flux', 'ap09FluxErr', 'ap09Flux_flag', 'ap12Flux', 'ap12FluxErr', 'ap12Flux_flag', 'ap17Flux', 'ap17FluxErr', 'ap17Flux_flag', 'ap25Flux', 'ap25FluxErr', 'ap25Flux_flag', 'ap35Flux', 'ap35FluxErr', 'ap35Flux_flag', 'ap50Flux', 'ap50FluxErr', 'ap50Flux_flag', 'ap70Flux', 'ap70FluxErr', 'ap70Flux_flag', 'sky', 'skyErr', 'psfFlux', 'psfFluxErr', 'ixx', 'iyy', 'ixy', 'ixxPSF', 'iyyPSF', 'ixyPSF', 'ixxDebiasedPSF', 'iyyDebiasedPSF', 'ixyDebiasedPSF', 'gaussianFlux', 'gaussianFluxErr', 'extendedness', 'sizeExtendedness', 'blendedness_abs', 'blendedness_flag', 'blendedness_flag_noCentroid', 'blendedness_flag_noShape', 'apFlux_12_0_flag', 'apFlux_12_0_flag_apertureTruncated', 'apFlux_12_0_flag_sincCoeffsTruncated', 'apFlux_12_0_instFlux', 'apFlux_12_0_instFlux

### 4 Image

#### 4.1. Sélectionner un détecteur commun aux deux datasets
Dans ton notebook, commence par afficher les dataId disponibles :

In [30]:
import pandas as pd

refs_img = list(
    butler.registry.queryDatasets(
        datasetType="post_isr_image",
        collections=collection_path,
    )
)

refs_cat = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Images :", len(refs_img))
print("Catalogues :", len(refs_cat))

print("\nPremières images :")
for ref in refs_img[:5]:
    print(ref.dataId)

print("\nPremiers catalogues :")
for ref in refs_cat[:5]:
    print(ref.dataId)

Images : 179
Catalogues : 179

Premières images :
{instrument: 'LSSTCam', detector: 87, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 145, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 55, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 41, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 59, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}

Premiers catalogues :
{instrument: 'LSSTCam', detector: 47, visit: 2025123100242, band: 'i', day_obs: 20251231, physical_filter: 'i_39'}
{instrument: 'LSSTCam', detector: 56, visit: 202512

Repère un couple qui possède le même visit et le même detector.
#### 4.2. Charger l'image et le catalogue
Une fois le dataId commun identifié, sélectionne les deux références correspondantes. Par exemple :

In [34]:
print("IMAGE")
print(refs_img[0].dataId)
print("Dimensions :", list(refs_img[0].dataId.dimensions.names))

print("\nCATALOGUE")
print(refs_cat[0].dataId)
print("Dimensions :", list(refs_cat[0].dataId.dimensions.names))

IMAGE
{instrument: 'LSSTCam', detector: 87, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
Dimensions : ['band', 'instrument', 'day_obs', 'detector', 'group', 'physical_filter', 'exposure']

CATALOGUE
{instrument: 'LSSTCam', detector: 47, visit: 2025123100242, band: 'i', day_obs: 20251231, physical_filter: 'i_39'}
Dimensions : ['band', 'instrument', 'day_obs', 'detector', 'physical_filter', 'visit']


In [35]:
detectors_img = {
    r.dataId["detector"]
    for r in refs_img
}

detectors_cat = {
    r.dataId["detector"]
    for r in refs_cat
}

common_detectors = sorted(detectors_img & detectors_cat)

print("Nombre de détecteurs communs :", len(common_detectors))
print("Détecteurs :", common_detectors)

Nombre de détecteurs communs : 179
Détecteurs : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 121, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 162, 163, 164, 165, 166, 167, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187]


In [36]:
detector_id = 47  # À remplacer par le détecteur choisi

ref_img = next(
    r for r in refs_img
    if r.dataId["detector"] == detector_id
)

ref_cat = next(
    r for r in refs_cat
    if r.dataId["detector"] == detector_id
    and r.dataId["visit"] == ref_img.dataId["exposure"]
)

exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Image :", ref_img.dataId)
print("Catalogue :", ref_cat.dataId)
print("Dimensions image :", exposure.image.array.shape)
print("Type catalogue :", type(catalog))

Image : {instrument: 'LSSTCam', detector: 47, exposure: 2025123100242, band: 'i', day_obs: 20251231, group: '2026-01-01T05:18:50.970', physical_filter: 'i_39'}
Catalogue : {instrument: 'LSSTCam', detector: 47, visit: 2025123100242, band: 'i', day_obs: 20251231, physical_filter: 'i_39'}
Dimensions image : (4004, 4096)
Type catalogue : <class 'astropy.table.table.Table'>


#### 4.3 Charger les deux données

In [37]:
exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Type image :", type(exposure))
print("Type catalogue :", type(catalog))
print("Dimensions image :", exposure.image.array.shape)

Type image : <class 'lsst.afw.image._exposure.ExposureF'>
Type catalogue : <class 'astropy.table.table.Table'>
Dimensions image : (4004, 4096)


In [38]:
print(catalog.schema if hasattr(catalog, "schema") else catalog.colnames)

['coord_ra', 'coord_dec', 'parentSourceId', 'x', 'y', 'xErr', 'yErr', 'ra', 'dec', 'raErr', 'decErr', 'ra_dec_Cov', 'calibFlux', 'calibFluxErr', 'ap03Flux', 'ap03FluxErr', 'ap03Flux_flag', 'ap06Flux', 'ap06FluxErr', 'ap06Flux_flag', 'ap09Flux', 'ap09FluxErr', 'ap09Flux_flag', 'ap12Flux', 'ap12FluxErr', 'ap12Flux_flag', 'ap17Flux', 'ap17FluxErr', 'ap17Flux_flag', 'ap25Flux', 'ap25FluxErr', 'ap25Flux_flag', 'ap35Flux', 'ap35FluxErr', 'ap35Flux_flag', 'ap50Flux', 'ap50FluxErr', 'ap50Flux_flag', 'ap70Flux', 'ap70FluxErr', 'ap70Flux_flag', 'sky', 'skyErr', 'psfFlux', 'psfFluxErr', 'ixx', 'iyy', 'ixy', 'ixxPSF', 'iyyPSF', 'ixyPSF', 'ixxDebiasedPSF', 'iyyDebiasedPSF', 'ixyDebiasedPSF', 'gaussianFlux', 'gaussianFluxErr', 'extendedness', 'sizeExtendedness', 'blendedness_abs', 'blendedness_flag', 'blendedness_flag_noCentroid', 'blendedness_flag_noShape', 'apFlux_12_0_flag', 'apFlux_12_0_flag_apertureTruncated', 'apFlux_12_0_flag_sincCoeffsTruncated', 'apFlux_12_0_instFlux', 'apFlux_12_0_instFlux